# HubbardBath 06 — Hamiltonian simulation

Milestone 05 gave us an explicit four-qubit Pauli Hamiltonian. Here we use it to approximate

$$U(\tau)=e^{-iH\tau}$$

with first-order Lie–Trotter and second-order symmetric product formulas, and compare both against exact dense evolution.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

candidates=[Path.cwd(),Path.cwd().parent,Path.cwd()/'HubbardBath']
for c in candidates:
    if (c/'src'/'simulation.py').exists():
        repo_root=c.resolve(); break
else:
    raise FileNotFoundError('Run from the HubbardBath repository')
if str(repo_root) not in sys.path:
    sys.path.insert(0,str(repo_root))

from src.qubit_mapping import remove_zero_terms, two_site_hubbard_pauli_terms
from src.simulation import (
    computational_basis_state, convergence_exponent,
    exact_propagator, first_order_trotter, operator_norm_error,
    second_order_trotter, simulation_metrics
)


## 1. Build the four-qubit Hubbard Hamiltonian

We use the same $U/t=4$ model from the earlier milestones.

In [ ]:
terms=remove_zero_terms(two_site_hubbard_pauli_terms(t=1.0,U=4.0,mu=0.0))
print('Non-zero Pauli terms:',len(terms))
for label,coeff in sorted(terms.items()):
    print(f'{coeff:+.3f}  {label}')


## 2. Exact vs approximate evolution

For each Trotter step count $r$, we compare the product-formula propagator against the exact dense matrix exponential.

In [ ]:
tau=1.5
steps=np.array([1,2,4,8,16,32,64])
U_exact=exact_propagator(terms,tau)
first=[]; second=[]
for r in steps:
    first.append(operator_norm_error(first_order_trotter(terms,tau,int(r)),U_exact))
    second.append(operator_norm_error(second_order_trotter(terms,tau,int(r)),U_exact))
first=np.array(first); second=np.array(second)
p1=convergence_exponent(steps,first,tail_points=4)
p2=convergence_exponent(steps,second,tail_points=4)
print('First-order tail exponent:',p1)
print('Second-order tail exponent:',p2)


In [ ]:
plt.figure(figsize=(7,4.8))
plt.loglog(steps,first,marker='o',label=f'first order, slope={p1:.2f}')
plt.loglog(steps,second,marker='o',label=f'second order, slope={p2:.2f}')
plt.xlabel('Trotter steps')
plt.ylabel(r'$||U_{PF}-U||_2$')
plt.title(r'Hubbard simulation error at $U/t=4$, $\tau t=1.5$')
plt.legend(); plt.grid(alpha=0.2); plt.tight_layout(); plt.show()


## 3. State-specific error

We also evolve the doublon state $|1100\rangle$ and measure infidelity relative to exact evolution.

In [ ]:
psi0=computational_basis_state('1100')
rows=[]
for r in steps:
    rows.append(simulation_metrics(terms,tau,int(r),initial_state=psi0))
first_inf=np.array([x['first_order_state_infidelity'] for x in rows])
second_inf=np.array([x['second_order_state_infidelity'] for x in rows])
plt.figure(figsize=(7,4.8))
plt.loglog(steps,first_inf,marker='o',label='first order')
plt.loglog(steps,second_inf,marker='o',label='second order')
plt.xlabel('Trotter steps'); plt.ylabel('Doublon-state infidelity')
plt.title('State-specific simulation error'); plt.legend(); plt.grid(alpha=0.2)
plt.tight_layout(); plt.show()


## Interpretation

The point is not merely that the error decreases. We test the expected asymptotic behaviour: roughly $1/r$ for first order and $1/r^2$ for second order for this finite decomposition.

**Next:** reuse the same coefficients $\alpha_j$ to construct an LCU/block encoding of $H/\alpha$.